# Lab 5 — Beat the baseline, or explain why you did not
### EP0418 · Topic 6 · 3 practical hours

**You need:** `ep0418.py` and `sg_taxi_hourly.csv` (or your own track's data).

---
The rule for today, and for CA3:

> **Compute the baseline first. Then build the model. Then compare at the same
> horizon.**

Doing it in that order is what stops you accidentally choosing whichever
comparison flatters you.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

## Set up Gemini for this notebook (once, 30 seconds)

Open Gemini (the spark icon at the bottom of the notebook). In its chat box, click the **sliders icon** next to **+**, choose **Configure**, paste the text below and save. Leave the mode on **Default**. Colab keeps the instructions with this notebook, so every answer follows the course rules.

```
I am new to coding. Explain any code you write in plain words: what it does to my data.
Before writing code for a bigger task, tell me your plan in plain words.
Never delete or drop rows unless I ask. If a step would remove rows, say so first.
After changing the data, show the row count before and after, and the smallest and largest values.
Codes such as -999 mean a missing reading. Point them out; do not treat them as real numbers.
Do not change any line that starts with check. Those are the lab's locked checks.
If you are not sure, say so. Do not guess.
```

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True

CSV = "sg_taxi_hourly.csv"
fetch(CSV)

raw = pd.read_csv(CSV, parse_dates=["timestamp"])
s = raw.set_index("timestamp").taxi_count.resample("h").mean()
print(f"{len(s):,} hourly slots, mean {s.mean():.0f}")

# Placeholders so the checks give you a readable message rather than a
# NameError if you run a later cell before filling in an earlier one.
H = H_baseline = best = mae_model = mae_persistence = mae_snaive = pred = None

## Task 1 — Commit to a horizon FIRST

Before any modelling: how far ahead does your system actually need to see?

Write it in the cell below. You will be held to it.

**My forecast horizon:** ___ hours

**Why my system needs exactly that:**

## Task 2 — Score both baselines at your horizon

For a horizon of `h`, persistence may only use data from `h` hours ago.
Seasonal-naive always looks back 168 hours.

In [ ]:
H = ...    # TODO(you): your horizon in hours, from Task 1

# TODO(you): build the two baseline predictions at horizon H
persistence = ...
snaive      = ...

# TODO(you): MAE of each, using only rows where actual AND prediction exist
mae_persistence = ...
mae_snaive      = ...

if filled(H, persistence, snaive, mae_persistence, mae_snaive):
    best = min(mae_persistence, mae_snaive)
    H_baseline = H          # the horizon these baselines were scored at
    print(f"horizon {H}h")
    print(f"  persistence    MAE {mae_persistence:,.1f}")
    print(f"  seasonal-naive MAE {mae_snaive:,.1f}")
    print(f"  >>> the number to beat is {best:,.1f}")

## Task 3 — Now build a model

Run the install cell first. If it ends with a red error mentioning `prophet`,
that is not your mistake: use StatsForecast instead (second prompt below).

Score the model only on hours it did not see. Fit on the first six weeks and
forecast the last two, so that the comparison with the baseline is fair.

> Fit a Prophet model to the first six weeks of this hourly series `s` with
> daily and weekly seasonality. Forecast the last two weeks, H hours ahead at a
> time. Return `pred`, a Series aligned to `s` for those two weeks. Then show me
> how many hours were forecast and the first and last timestamps.

> Using StatsForecast, fit an AutoETS model on the first six weeks of the hourly
> series `s` and forecast the last two weeks H hours ahead. Return `pred`, a
> Series aligned to `s` for those two weeks. Then show me how many hours were
> forecast and the first and last timestamps.

In [ ]:
# Run this once. It takes a minute. If it fails, use StatsForecast (see above).
!pip -q install prophet 2>/dev/null || pip -q install statsforecast

In [ ]:
# TODO(you): fit a model and produce `pred` aligned to `s`
pred = ...

# TODO(you): MAE of your model at the SAME horizon
mae_model = ...

# ---- verification contract — do not edit ----
check.same_horizon(H, H_baseline, unit="h")
check.compare_to_baseline(mae_model, best, metric="MAE")
check.report()

### If you got a NOTE rather than a PASS

Your model did not beat the baseline. **That is a legitimate result and it costs
you no marks.**

What costs marks is hiding it, or quietly re-scoring the baseline at a different
horizon until you win.

Write down what you think the baseline is capturing that your model is not.

## Task 4 — Plot the residuals

Subtract your prediction from the actual and plot what is left.

Flat noise means the cycle is fully captured. **Structure in the residuals** — a
particular weekday, a spike at a particular hour — is the feature you are missing.

In [ ]:
# TODO(you): plot the residuals, and their mean by hour-of-day
...

---
## Read before you run

A student chose a horizon of 6 hours. Their assistant scored the persistence baseline like this:

**Do not run it yet.** Read it, answer the two questions in the cell below,
and only then remove the `#` and run it to see whether you were right.

How many hours ahead is this baseline guessing? The model will be scored 6 hours ahead. Does this make the baseline easier or harder to beat than it should be?

Write down the persistence MAE at 1 hour and at 6 hours from your own results, and which one this code reports.

In [ ]:
# --- AI-generated. Read it first. ---
# H = 6
# persistence = s.shift(1)
# mae_persistence = (s - persistence).abs().mean()

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Before you leave

- Your baseline number, written into `verification_log.md`. You will quote it in
  CA3.
- One sentence: did your model beat it, at what horizon, and if not — why not?
- A **Decisions** line: the baseline and horizon you chose, and why that one.